# LSTM for Time Series Classification

## Imports

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from pytorch_lightning import Trainer, seed_everything

# maintaining seed for reproducibility
seed_everything(42)

from data_toolkit.models.lstm import (
    LSTMClassificationPredictor, 
    TimeSeriesClassificationDataModule
    )

## Helper Functions

In [ ]:
def generate_classification_data(n_samples, seq_len, n_features, n_classes):
    sequences = []
    for _ in range(n_samples):
        sequence = pd.DataFrame(np.random.randn(seq_len, n_features))
        label = np.random.randint(0, n_classes)
        sequences.append((sequence, label))
    return sequences

n_samples = 500
seq_len = 20
n_features = 5
n_classes = 3

### Loading data

In [ ]:
data = generate_classification_data(n_samples, seq_len, n_features, n_classes)
train_data, test_data = train_test_split(data, test_size=0.2)
val_data, test_data = train_test_split(test_data, test_size=0.5)

### Initializing the dataset and the model

In [ ]:
dm = TimeSeriesClassificationDataModule(
    train_sequences=train_data,
    val_sequences=val_data,
    test_sequences=test_data,
    batch_size=32
)
model = LSTMClassificationPredictor(n_features=n_features, n_classes=n_classes)

### Training the model

In [ ]:
# Training the model
trainer = Trainer(max_epochs=5, accelerator='auto', devices=1)
trainer.fit(model, datamodule=dm)
trainer.test(model, datamodule=dm)